In [1]:
# Install the excel reader engine if you don't have it yet
!pip install openpyxl

import pandas as pd

# Load the local file 
print("Loading data from your local folder... This might take around 30 seconds.")
df = pd.read_excel("C:/Users/kingk/Documents/Data_Project/Online Retail.xlsx", engine="openpyxl")

print(f"Data Loaded successfully! Shape: {df.shape}")

# Let's see the first 5 rows
df.head()

Loading data from your local folder... This might take around 30 seconds.
Data Loaded successfully! Shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:
# 1. Check for missing values in critical business columns
print("--- MISSING VALUES PER COLUMN ---")
print(df[['CustomerID', 'Description']].isnull().sum())
print("\n")

# 2. Check for negative quantities (Cancellations/Returns)
negative_qty = df[df['Quantity'] < 0].shape[0]
print("--- TRANSACTION ANOMALIES ---")
print(f"Number of rows with negative quantities: {negative_qty}")

# 3. Check for non-product transactions in the Description
print("\n--- SAMPLE OF GHOST/NON-PRODUCT ITEMS ---")

ghost_items = df[
    df['StockCode'].astype(str).str.contains(r'^[A-Za-z]', regex=True, na=False)
]

print(ghost_items['Description'].unique()[:5])

--- MISSING VALUES PER COLUMN ---
CustomerID     135080
Description      1454
dtype: int64


--- TRANSACTION ANOMALIES ---
Number of rows with negative quantities: 10624

--- SAMPLE OF GHOST/NON-PRODUCT ITEMS ---
['POSTAGE' 'Discount' 'CARRIAGE' 'DOTCOM POSTAGE' 'Manual']


Now, For Data Cleaning & Revenue Optimization

In [3]:
# 1. Remove rows where CustomerID is missing (we cannot track behavior without a unique ID)
df_clean = df.dropna(subset=['CustomerID']).copy()

# 2. Convert CustomerID from a decimal (float) to an integer for professional clean formatting
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

# 3. Create a clean subset of successful purchases by filtering out negative quantities (returns)
df_purchases = df_clean[df_clean['Quantity'] > 0].copy()

# 4. Filter out operational "ghost" rows (like POSTAGE or manual fees) by ensuring StockCodes are standard inventory numbers
# Standard items usually don't start with letters like 'POST', 'D', or 'M'
df_purchases = df_purchases[~df_purchases['StockCode'].astype(str).str.contains('^[A-Za-z]', regex=True, na=False)]

# 5. Create our primary metric: Total Revenue per line item (Quantity multiplied by the Unit Price)
df_purchases['LineTotal'] = df_purchases['Quantity'] * df_purchases['UnitPrice']

# Verify our progress by checking the new dataset size and column structure
print(f"Cleaned dataset shape: {df_purchases.shape}")
df_purchases.head()

Cleaned dataset shape: (396370, 9)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,LineTotal
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


Preparing the RFM Foundations

In [4]:
# 1. Ensure the InvoiceDate column is treated as a true datetime object by Python
df_purchases['InvoiceDate'] = pd.to_datetime(df_purchases['InvoiceDate'])

# 2. Establish our 'snapshot date' by finding the maximum date in the dataset and adding 1 day
snapshot_date = df_purchases['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"Snapshot Date (Our simulated 'Today'): {snapshot_date.date()}\n")

# 3. Aggregate data by CustomerID to calculate individual Recency, Frequency, and Monetary values
rfm = df_purchases.groupby('CustomerID').agg({
    # Recency: Calculate the number of days between the snapshot date and the customer's latest purchase
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    
    # Frequency: Count the number of unique invoices (orders) the customer placed
    'InvoiceNo': 'nunique',
    
    # Monetary: Sum up the total line item spend to get the lifetime value of the customer
    'LineTotal': 'sum'
}).reset_index()

# 4. Rename the columns to professional business terminology
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# 5. Filter out extreme edge-case rows where Monetary value is zero to keep our financial metrics accurate
rfm = rfm[rfm['Monetary'] > 0]

# Display the final shape of our customer metrics matrix and the first 5 customer summaries
print(f"RFM Matrix generated for {rfm.shape[0]} unique customers.")
rfm.head()

Snapshot Date (Our simulated 'Today'): 2011-12-10

RFM Matrix generated for 4334 unique customers.


,CustomerID,Recency,Frequency,Monetary
0,12346,326,1,77183.60
1,12347,2,7,4310.00
2,12348,75,4,1437.24
3,12349,19,1,1457.55
4,12350,310,1,294.40


Creating RFM Scores (Quantiles)

In [5]:
# 1. Use pandas 'qcut' to divide Recency into 5 equal percentage groups (quintiles)
# Note: For Recency, a smaller number of days is BETTER, so labels are inverted [5, 4, 3, 2, 1]
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1])

# 2. Use 'qcut' to divide Frequency into 5 equal groups
# Note: Higher frequency is BETTER, so labels are standard [1, 2, 3, 4, 5]
# We use rank(method='first') because many customers have identical low frequencies
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])

# 3. Use 'qcut' to divide Monetary into 5 equal groups
# Note: Higher total spend is BETTER, so labels are standard [1, 2, 3, 4, 5]
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=5, labels=[1, 2, 3, 4, 5])

# 4. Combine the three individual scores into a single string score (e.g., "555" or "111")
rfm['RFM_Cell'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

# 5. Display the newly scored customer dataframe
print("RFM Scores successfully generated!")
rfm[['CustomerID', 'Recency', 'R_Score', 'Frequency', 'F_Score', 'Monetary', 'M_Score', 'RFM_Cell']].head()

RFM Scores successfully generated!


,CustomerID,Recency,R_Score,Frequency,F_Score,Monetary,M_Score,RFM_Cell
0,12346,326,1,1,1,77183.60,5,115
1,12347,2,5,7,5,4310.00,5,555
2,12348,75,2,4,4,1437.24,4,244
3,12349,19,4,1,1,1457.55,4,414
4,12350,310,1,1,1,294.40,2,112


In [6]:
import numpy as np

# 1. Define a clear business mapping dictionary based on Recency and Frequency scores
# This translates mathematical combinations into clear business definitions
segs = {
    r'[4-5][4-5]': 'Champions',            # Bought recently, buy frequently
    r'[2-4][3-5]': 'Loyal Customers',      # Regular customers with good frequency
    r'[4-5][1-2]': 'Recent Customers',     # New shoppers, haven't bought often yet
    r'[3-4][1-2]': 'Promising',            # Early stage shoppers, showing potential
    r'[2-3][2-3]': 'Customers Needing Attention', # Average recency and frequency; could slip away
    r'[1-2][3-4]': 'At Risk',              # Used to buy often, but haven't returned recently
    r'1[4-5]': 'Can\'t Lose Them',         # High-frequency historical shoppers who have stopped completely
    r'1[1-2]': 'Lost'                      # Lowest scores across the board; inactive
}

# 2. Combine the R and F scores into a 2-digit string to apply our mapping dictionary
rfm['Reg_Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str)

# 3. Apply the segmentation definitions to the dataset using the mapping rules
rfm['Segment'] = rfm['Reg_Segment'].replace(segs, regex=True)

# 4. Clean up the dataframe by dropping the temporary mapping column
rfm = rfm.drop(columns=['Reg_Segment'])

# 5. Display a high-level corporate summary showing the size of each customer segment
print("--- CUSTOMER SEGMENT COUNT SUMMARY ---")
print(rfm['Segment'].value_counts())

# Save the file locally so we can import it into SQL next
rfm.to_csv("RFM_Customer_Segments.csv", index=False)
print("\nSuccess! Saved segmented data as 'RFM_Customer_Segments.csv'")

--- CUSTOMER SEGMENT COUNT SUMMARY ---
Segment
Champions                      1136
Loyal Customers                1129
Lost                            669
Promising                       347
Recent Customers                319
Customers Needing Attention     200
21                              199
At Risk                         185
53                              137
Can't Lose Them                  13
Name: count, dtype: int64

Success! Saved segmented data as 'RFM_Customer_Segments.csv'


In [12]:
import urllib.parse
from sqlalchemy import create_engine
import pandas as pd

# 1. Server configuration parameters
USER = "root"
PASSWORD = "Karsten@SQL101"  #-- Put your actual MySQL password string here
HOST = "localhost"
PORT = "3306"
DATABASE = "retail_analytics"

# 2. Secure parse and connection instantiation
safe_password = urllib.parse.quote_plus(PASSWORD)
connection_string = f"mysql+pymysql://{USER}:{safe_password}@{HOST}:{PORT}/{DATABASE}"
engine = create_engine(connection_string)

# 3. Align transactional data tables with verified customer lists
valid_customer_ids = rfm['CustomerID'].unique()
df_products_upload = df_purchases[['StockCode', 'Description']].drop_duplicates(subset=['StockCode'])
df_sales_upload = df_purchases[['InvoiceNo', 'StockCode', 'CustomerID', 'Quantity', 'InvoiceDate', 'UnitPrice', 'LineTotal']]
df_sales_fixed = df_sales_upload[df_sales_upload['CustomerID'].isin(valid_customer_ids)].copy()

# 4. Stream data directly to tables
print("Uploading dim_customers...")
rfm.to_sql(name='dim_customers', con=engine, if_exists='append', index=False)

print("Uploading dim_products...")
df_products_upload.to_sql(name='dim_products', con=engine, if_exists='append', index=False)

print("Uploading aligned fact_sales...")
df_sales_fixed.to_sql(name='fact_sales', con=engine, if_exists='append', index=False)

print("\nSuccess! The entire Star Schema database is populated and clean.")

Uploading dim_customers...
Uploading dim_products...
Uploading aligned fact_sales...

Success! The entire Star Schema database is populated and clean.


In [13]:
# 1. Create a clean master directory of CustomerIDs and their corresponding Country from our cleaned sales data
customer_countries = df_purchases[['CustomerID', 'Country']].drop_duplicates(subset=['CustomerID'])

# 2. Drop the empty/incomplete country column from our current scored rfm table
if 'Country' in rfm.columns:
    rfm = rfm.drop(columns=['Country'])

# 3. Use a left merge to stitch the real country text back onto our RFM matrix
rfm = pd.merge(rfm, customer_countries, on='CustomerID', how='left')

# 4. Rearrange columns so 'Country' is right next to CustomerID, matching our SQL architecture setup
column_order = ['CustomerID', 'Country', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'RFM_Cell', 'Segment']
rfm = rfm[column_order]

print("Country data successfully restored in Python! Quick preview:")
rfm[['CustomerID', 'Country', 'Segment']].head()

Country data successfully restored in Python! Quick preview:


,CustomerID,Country,Segment
0,12346,United Kingdom,Lost
1,12347,Iceland,Champions
2,12348,Finland,Loyal Customers
3,12349,Italy,Recent Customers
4,12350,Norway,Lost


In [14]:
# Stream our newly matched table into MySQL dim_customers
print("Re-uploading clean dim_customers with location data...")
rfm.to_sql(name='dim_customers', con=engine, if_exists='append', index=False)
print("Upload Complete!")

Re-uploading clean dim_customers with location data...
Upload Complete!
